# Residual Block

A residual block computes:

`y = F(x) + S(x)`

Using `N x C x H x W` tensors, `S(x) = x` when input and output shapes match. If spatial size or channel count changes, `S(x)` must transform the input. Element-wise addition requires identical tensor shapes.

**Objective:** Complete a residual block that automatically uses an identity shortcut or a projection shortcut.


In [6]:
import torch
import torch.nn as nn


torch.manual_seed(7)


class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # TODO 1: Primera convolución 3x3
        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False,
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        # TODO 2: Conexión de atajo (Shortcut)
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.relu(residual)
        residual = self.conv2(residual)
        residual = self.bn2(residual)

        shortcut = self.shortcut(x)

        # TODO 3: Suma elemento a elemento y activación ReLU
        out = residual + shortcut
        return self.relu(out)

In [7]:
x_identity = torch.randn(4, 32, 64, 64)

identity_block = ResidualBlock(
    in_channels=32,
    out_channels=32,
    stride=1,
)

output = identity_block(x_identity)

print(f"Input shape: {x_identity.shape}")
print(f"Output shape: {output.shape}")
print(f"Shortcut module: {identity_block.shortcut}")

assert output.shape == x_identity.shape
assert isinstance(identity_block.shortcut, nn.Identity)

print("Question: Why can the input be added directly to the residual path here?")


Input shape: torch.Size([4, 32, 64, 64])
Output shape: torch.Size([4, 32, 64, 64])
Shortcut module: Identity()
Question: Why can the input be added directly to the residual path here?


In [8]:
x_projection = torch.randn(
    4,
    32,
    64,
    64,
    requires_grad=True,
)

projection_block = ResidualBlock(
    in_channels=32,
    out_channels=64,
    stride=2,
)

output = projection_block(x_projection)

print(f"Input shape: {x_projection.shape}")
print(f"Residual block output shape: {output.shape}")
print(f"Shortcut module: {projection_block.shortcut}")

expected_shape = torch.Size([4, 64, 32, 32])
assert output.shape == expected_shape

loss = output.mean()
loss.backward()

print(f"Input gradient norm: {x_projection.grad.norm().item():.6f}")
print("Reflection questions:")
print("1. Which dimensions changed?")
print("2. Why is nn.Identity() not enough here?")
print("3. What does x_projection.grad not being None show?")


Input shape: torch.Size([4, 32, 64, 64])
Residual block output shape: torch.Size([4, 64, 32, 32])
Shortcut module: Sequential(
  (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)
Input gradient norm: 0.001264
Reflection questions:
1. Which dimensions changed?
2. Why is nn.Identity() not enough here?
3. What does x_projection.grad not being None show?
